In [ ]:
# !poetry run jupyter lab --ServerApp.token='' --ServerApp.password=''


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
repo_root = pathlib.Path().resolve()  # adjust if you launched Jupyter in a subdir
# sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))



In [ ]:

from pprint import pprint
# from plainera_unacronym.nlp.execute import detect_and_extract
from plainera_unacronym.nlp.execute import detect_and_extract
import json

# 2) Sanity test (text path)
text = """Note that U.S. and U.K. dotted forms appear here but aren’t acronyms under our pattern. But the United Kingdom (UK) and USA should appear in this sentence."""
json_str = detect_and_extract(text)
pprint(json_str)   # pretty JSON in the cell


In [ ]:
from pprint import pprint
from plainera_unacronym.nlp.extraction import ExtractionConfig
from plainera_unacronym.nlp.extraction.engine.detect_flow import ExtractionFlow
from plainera_unacronym.nlp.common.types import DetectorConfig


def run_extraction(text: str):
    flow = ExtractionFlow(
        det_cfg=DetectorConfig(),
        ext_cfg=ExtractionConfig(),
        window_left=320,
        window_right=280,
    )

    det, extr, reports = flow.run(text)


    return det, extr, reports


In [ ]:
import textwrap
from plainera_unacronym.nlp.execute import detect_and_extract
from pprint import pprint
LONG_TEXT = textwrap.dedent("""
    The American Psychological Association (APA) publishes influential journals, while the American Planning Association (APA) guides urban development.
    In finance, the Consumer Price Index (CPI) tracks inflation, whereas in computing CPI often refers to cycles per instruction.
    Single Sign-On (SSO) simplifies access, and SSO stands for Single Sign-On across most IT platforms. Lightweight Directory Access Protocol (LDAP) integrates
    with Transport Layer Security (TLS) for secure binds in many NHS trusts. Jacob says, ALRIGHTY THEN!

    The Americans with Disabilities Act (ADA) ensures accessibility, but the American Dental Association (ADA) sets clinical guidelines.
    Corporate Social Responsibility (CSR) shapes strategy, whereas a Certificate Signing Request (CSR) kicks off PKI workflows.
    We ran GPU–accelerated jobs and compared GPU results to CPU baselines; R&D will review them alongside I/O traces and S&P 500 sector notes.

    The European Medicines Agency (EMA) approves drugs in the EU, while an exponential moving average (EMA) is a trading indicator.
    Centers for Disease Control and Prevention (CDC) issue guidance; in data engineering, Change Data Capture (CDC) drives downstream updates.
    Return on Investment (ROI) guides budgets, but Region of Interest (ROI) guides image processing.

    The Department of Energy (DOE) funds basic research, while Design of Experiments (DOE) structures trials.
    The Securities and Exchange Commission (SEC) oversees markets; the Southeastern Conference (SEC) organizes college sports.
    The Central Processing Unit (CPU) remains a baseline as Graphics Processing Units (GPU) scale out; Random Access Memory (RAM) capacity still gates workloads.
    “IT was tricky to reproduce” is just a sentence start, but IT (Information Technology) owns the SSO/LDAP stack.

    Digital Subscriber Line (DSL) brought early broadband; a Domain-Specific Language (DSL) made our pipeline concise.
    A Peripheral Component Interconnect (PCI) slot differs from Payment Card Industry (PCI) compliance.
    Earnings Per Share (EPS) moved after ISO-certified audits; Encapsulated PostScript (EPS) assets rendered crisply.
    We exported JSON and CSV snapshots; H2O chemistry demos stayed separate from MP3 decoding tests.

    The International Telecommunication Union (ITU) sets standards; the International Triathlon Union (ITU) runs competitions.
    The National Archives and Records Administration (NARA) preserves documents; the North American Retail Association (NARA) advocates for merchants.
    O’RAN specs advanced; USB-C hubs shipped. OK, we’ll regroup at 10:45 AM, and PM (Project Manager) will chair; later, PM stands for particulate matter.

    The British Standards Institution (BSI) audits suppliers; Business Systems Integration (BSI) teams coordinate ERP rollouts.
    Quality Assurance (QA) wrote test plans; Quality Control (QC) validated outputs.
    User Experience (UX) and User Interface (UI) workshops ran back-to-back; Estimated Time of Arrival (ETA) for the next build is 18:30.
    Note that U.S. and U.K. dotted forms appear here but aren’t acronyms under our pattern.

    The World Wide Web Consortium (W3C) advanced specs; HyperText Markup Language (HTML) docs and Cascading Style Sheets (CSS) were version-locked.
    The International Organization for Standardization (ISO) reviewed findings; the Insurance Services Office (ISO) published actuarial updates.
    The Federal Communications Commission (FCC) ruled on spectrum; Field-Programmable Gate Arrays (FPGA) sped up TLS offload.

    The International Criminal Court (ICC) issued guidance; in sports, the International Cricket Council (ICC) scheduled fixtures.
    The European Central Bank (ECB) raised rates; the Electronic Code Book (ECB) mode remained deprecated in crypto courses.
    Meanwhile, the North Atlantic Treaty Organization (NATO) met with the National Oceanic and Atmospheric Administration (NOAA) about satellite data.
    NASA’s outreach continues, while NASA events celebrate saxophone music in the North American Saxophone Alliance (NASA).
""").strip()

SHORT_TEXT = textwrap.dedent("""
    The International Criminal Court (ICC) issued guidance; in sports, the International Cricket Council (ICC) scheduled fixtures.
    The European Central Bank (ECB) raised rates; the Electronic Code Book (ECB) mode remained deprecated in crypto courses.
    Meanwhile, the North Atlantic Treaty Organization (NATO) met with the National Oceanic and Atmospheric Administration (NOAA) about satellite data.
    NASA’s outreach continues, while NASA events celebrate saxophone music in the North American Saxophone Alliance (NASA).
""").strip()



In [ ]:
test_1 = "The Single sign-on (SSO) is enabled."
test_2 = "Natural language processing (NLP) is used."
det, ext, report = run_extraction(test_2)
pprint(det)
pprint(ext)
# pprint(report)

In [ ]:
from plainera_unacronym.nlp.extraction.engine.detect_flow import ExtractionFlow
from plainera_unacronym.nlp.extraction import ExtractionConfig
from plainera_unacronym.nlp.common.types import DetectorConfig


def run_extraction(text: str):
    flow = ExtractionFlow(
        det_cfg=DetectorConfig(),
        ext_cfg=ExtractionConfig(window_chars=360, margin_threshold=0.15),
        window_left=320,
        window_right=280,
    )
    det, extr, reports = flow.run(text)
    return det, extr, reports


def _picked_def(extr, key: str):
    """Return extracted definition for acronym key if present, else None."""
    pick = extr.picks.get(key)
    if pick is None:
        return None
    return pick.definition


# 1) Lower-case tokens should be preserved (no truncation)
det, extr, reports = run_extraction("Single sign-on (SSO) is enabled.")
assert _picked_def(extr, "SSO") in {"Single sign-on"}, extr.picks.get("SSO")

# det, extr, reports = run_extraction("single sign-on (SSO) is enabled.")
# assert _picked_def(extr, "SSO") in {"single sign-on"}, extr.picks.get("SSO")


det, extr, reports = run_extraction("return on investment (ROI) is tracked.")
assert _picked_def(extr, "ROI") == "return on investment", extr.picks.get("ROI")

# 2) Should NOT hallucinate a definition when acronym is used but not defined
det, extr, reports = run_extraction("We discussed options and agreed on the approach (SLA) yesterday.")
assert extr.picks.get("SLA") is None, extr.picks.get("SLA")

# 3) Proper noun should still work
det, extr, reports = run_extraction("National Health Service (NHS) guidelines apply.")
assert _picked_def(extr, "NHS") == "National Health Service", extr.picks.get("NHS")

print("All assertions passed.")


## tier one importance

In [ ]:
TIER_ONE_TEXT = textwrap.dedent("""We store authentication using JSON Web Tokens. JWT is issued after login and saved in a secure cookie.

Single sign-on (SSO) is enabled for enterprise accounts. This is different from standard login, but users sometimes confuse the terms.

Our encryption is end-to-end (E2E) for messages sent between clients.

Natural language processing (NLP) is used to detect entities, but the NLP output can be noisy.

Personal protective equipment (PPE, required on site) must be worn in the laboratory at all times.

The Chief Executive Officer (CEO) approved the new security policy and requested weekly reporting.

In this document, the National Health Service (NHS) is referenced frequently as a case study.

The service-level agreement, abbreviated as SLA, defines uptime commitments for the platform.""")

In [ ]:
run_extraction(TIER_ONE_TEXT)

## tier two importance

In [ ]:
TIER_TWO_TEXT = textwrap.dedent("""The organisation tracks key performance indicators (KPIs) at the team and department level.

The NHS’s internal documentation contains many SOPs, but those SOPs are not always followed consistently.

The app supports machine learning / ML features for classification and ranking.

LDAP, a directory protocol, is used to authenticate some legacy users.

Service Level Objectives (SLOs) and Service Level Indicators (SLIs) are monitored continuously.

For availability reporting, we use the following mapping:
NHS — National Health Service
ROI — return on investment
COGS — cost of goods sold

Some teams refer to the “NHS” in quotes to emphasise the name in policy documents.
""")

In [ ]:
run_extraction(TIER_TWO_TEXT)

## tier one and two mixed importance

In [ ]:
TIER_MIXED_TEXT = textwrap.dedent("""We store authentication using JSON Web Tokens. JWT is issued after login and saved in a secure cookie. Some services also use single sign-on (SSO) for enterprise accounts.

The Chief Executive Officer (CEO) approved the security policy. Personal protective equipment (PPE, required on site) must be worn in the laboratory.

Our encryption is end-to-end (E2E). The service-level agreement, abbreviated as SLA, defines the uptime commitments, and these are tracked via Service Level Objectives (SLOs) and Service Level Indicators (SLIs).

Natural language processing (NLP) is used to extract entities. The NLP output can be noisy, especially when users paste bullet lists.

In this document, the National Health Service (NHS) is referenced frequently. The NHS’s internal documentation also contains SOPs, but the SOPs are not always followed.

Some teams refer to machine learning / ML features when discussing ranking and classification. LDAP, a directory protocol, is still used for a subset of legacy accounts.

For reporting, the organisation tracks key performance indicators (KPIs). In some documents you will see dash mappings like:
NHS — National Health Service
COGS — cost of goods sold""")

In [ ]:
run_extraction(TIER_MIXED_TEXT)

In [ ]:

from plainera_unacronym.nlp.extraction import ExtractionConfig
from plainera_unacronym.nlp.extraction.engine.detect_flow import ExtractionFlow
from plainera_unacronym.nlp.common.types import DetectorConfig

flow = ExtractionFlow(
    det_cfg=DetectorConfig(),
    ext_cfg=ExtractionConfig(window_chars=360, margin_threshold=0.15),
    window_left=320, window_right=280,
)
det, extr, reports = flow.run(LONG_TEXT)

print("********************"*10)
print("DET")
# pprint(det)
pprint(extr)
# pprint(reports)

In [ ]:
 # tests/test_acronyms_rd.py
from pprint import pprint
text = "We’ll loop in R&D after the NHS workshop. IT (Information Technology) leads."
pprint(detect_and_extract(text))
# data = json.loads(run_detection(text, as_json=True))
# keys = set(data["unique_acronyms"].keys())
# print(keys)
# assert "R&D" in keys
# assert data["unique_acronyms"]["R&D"]["confidence"] >= 0.60


In [ ]:
# --- setup (adjust path if needed) ---
import sys, pathlib, json

try:
    import plainera_unacronym  # noqa
except ModuleNotFoundError:
    repo_root = pathlib.Path().resolve()
    sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))

def detect(text: str, **kwargs):
    """Call your detector and return a parsed dict."""
    return detect_and_extract(text)

In [ ]:
# --- setup / imports ---
import sys, pathlib, json
from dataclasses import asdict, is_dataclass

try:
    import plainera_unacronym  # noqa
except ModuleNotFoundError:
    repo_root = pathlib.Path().resolve()
    sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))

# make sure we import the function we actually call

# --- helpers ---
def _to_dict(obj):
    """Recursively convert dataclasses into dicts; pass other types through."""
    if is_dataclass(obj):
        return asdict(obj)
    if isinstance(obj, (list, tuple)):
        return [_to_dict(x) for x in obj]
    if isinstance(obj, dict):
        return {k: _to_dict(v) for k, v in obj.items()}
    return obj

def _normalize_output(det_res, ext_res):
    """Shape the output to what tests expect."""
    det_d = _to_dict(det_res)   # {'unique_acronyms': {...}, 'occurrences': [...]}
    ext_d = _to_dict(ext_res)
    # If you want extraction fields available for later assertions, you can attach them:
    # det_d["extraction"] = _to_dict(ext_res)
    return det_d, ext_d

def detect(text: str, **kwargs):
    """Call your pipeline and return a plain dict suitable for the tests."""
    det_res, ext_res = detect_and_extract(text)
    return _normalize_output(det_res, ext_res)

# --- tiny test harness ---
PASSED = 0
FAILED = 0

def check(name: str, fn):
    global PASSED, FAILED
    try:
        fn()
        print(f"✅ {name}")
        PASSED += 1
    except AssertionError as e:
        print(f"❌ {name}: {e}")
        FAILED += 1
    except Exception as e:
        print(f"💥 {name}: {type(e).__name__}: {e}")
        FAILED += 1


def keys(d: dict) -> set[str]:
    return set(d["unique_acronyms"].keys())

def counts_by_acronym(d: dict) -> dict[str, int]:
    out = {}
    for o in d["occurrences"]:
        out[o["acronym"]] = out.get(o["acronym"], 0) + 1
    return out

# --- tests ---
def test_rd_detected_and_normalized():
    d = detect("We’ll loop in R & D after the NHS workshop.")
    assert "R&D" in keys(d), f"got {keys(d)}"
    assert d["unique_acronyms"]["R&D"]["confidence"] >= 0.60

def test_ok_and_am_drop_in_normal_prose():
    d = detect("OK, let's meet at 10:30 AM after lunch.")
    ks = keys(d)
    assert "OK" not in ks, f"OK leaked in: {ks}"
    assert "AM" not in ks, f"AM leaked in: {ks}"

def test_it_drops_as_pronoun_but_kept_with_definition():
    d1 = detect("IT was raining when we arrived.")
    assert "IT" not in keys(d1), f"pronoun IT should drop, got {keys(d1)}"

    d2 = detect("IT (Information Technology) owns the LDAP stack.")
    assert "IT" in keys(d2), f"IT with definition should keep, got {keys(d2)}"
    assert d2["unique_acronyms"]["IT"]["confidence"] >= 0.72

def test_stands_for_directional_rightward():
    d = detect("AM stands for amplitude modulation. OK, fine.")
    assert "AM" in keys(d), f"AM in definitional context should keep, got {keys(d)}"
    assert d["unique_acronyms"]["AM"]["confidence"] >= 0.72
    assert "OK" not in keys(d), f"OK should drop, got {keys(d)}"

def test_curly_apostrophe_and_hyphen_variants():
    d = detect("O’RAN and USB-C are on the agenda with the NHS.")
    ks = keys(d)
    assert "O'RAN" in ks, f"O'RAN missing, got {ks}"
    assert "USB-C" in ks, f"USB-C missing, got {ks}"
    assert "NHS" in ks

def test_mixed_alnum_kept():
    d = detect("H2O and MP3 appear in the doc.")
    ks = keys(d)
    assert "H2O" in ks, f"H2O missing, got {ks}"
    assert "MP3" in ks, f"MP3 missing, got {ks}"

def test_length_aware_threshold_blocks_bare_two_letter():
    d = detect("We will use AI and GPU for this.")
    ks = keys(d)
    assert "AI" not in ks, f"AI should drop under 2-letter threshold, got {ks}"
    assert "GPU" in ks, f"GPU should pass, got {ks}"


def test_company_suffixes_drop_without_definition():
    d = detect("Acme LTD and Example PLC signed the MOU.")
    ks = keys(d)
    assert "LTD" not in ks, f"LTD leaked in: {ks}"
    assert "PLC" not in ks, f"PLC leaked in: {ks}"
    # don't assert on MOU (might or might not be detected depending on your config)

def test_parallel_matches_serial_results():
    base = "We’ll loop in R&D after the NHS workshop. IT (Information Technology) leads. "
    big = base * 200  # large enough to trigger parallel path (if enabled)
    serial = detect(big, parallel=False)
    parallel = detect(big, parallel=True)
    assert keys(serial) == keys(parallel), f"unique sets differ: {keys(serial)} vs {keys(parallel)}"
    assert counts_by_acronym(serial) == counts_by_acronym(parallel), \
        f"occurrence counts differ: {counts_by_acronym(serial)} vs {counts_by_acronym(parallel)}"

# --- run all ---
tests = [
    ("R&D detected & normalized", test_rd_detected_and_normalized),
    ("OK/AM drop in prose", test_ok_and_am_drop_in_normal_prose),
    ("IT: pronoun drops, definition keeps", test_it_drops_as_pronoun_but_kept_with_definition),
    ("'stands for' directional", test_stands_for_directional_rightward),
    ("Curly apostrophe & hyphen", test_curly_apostrophe_and_hyphen_variants),
    ("Mixed alnum kept", test_mixed_alnum_kept),
    ("Length-aware threshold on 2-letter", test_length_aware_threshold_blocks_bare_two_letter),
    ("Company suffixes drop", test_company_suffixes_drop_without_definition),
    ("Parallel parity", test_parallel_matches_serial_results),
]

for name, fn in tests:
    check(name, fn)

print(f"\nSummary: {PASSED} passed, {FAILED} failed")


In [ ]:
from pprint import pprint
def test_big():
    base = (
            "In printing, PTO stands for "
            "a very, very long descriptive phrase that should be trimmed or rejected entirely "
            "depending on configuration and normalisation steps. "
            "Portable Document Format (PDF) is common."
        )
    d = detect(base)
    pprint(d)
    ks = keys(d)
    pprint(ks)

test_big()

In [ ]:
def test_numeric_leading_token_is_preserved_in_parenthetical_after():
    # Ensure numeric-leading tokens (e.g., 3M) are kept in the definition window
    text = "PF (3M Portable format) is a special case in this doc."
    extr = detect(text)
    pprint(extr)

test_numeric_leading_token_is_preserved_in_parenthetical_after()


In [ ]:
# === full-output wrapper (detect + extraction) ===
from dataclasses import asdict, is_dataclass
from plainera_unacronym.nlp.execute import detect_and_extract

def _to_dict(obj):
    if is_dataclass(obj):
        return asdict(obj)
    if isinstance(obj, (list, tuple)):
        return [_to_dict(x) for x in obj]
    if isinstance(obj, dict):
        return {k: _to_dict(v) for k, v in obj.items()}
    return obj

def detect_full(text: str, **kwargs) -> dict:
    det_res, ext_res = detect_and_extract(text)
    out = _to_dict(det_res)
    out["extraction"] = _to_dict(ext_res)
    return out

# --- helpers for these tests ---
def sense_ids(extraction: dict, acr: str) -> set[str]:
    return set(s["sense_id"] for s in extraction["senses_by_acronym"].get(acr, []))

def chosen_ids_for_acr(extraction: dict, acr: str) -> set[str | None]:
    return set(
        r.get("chosen_sense_id")
        for r in extraction.get("resolutions", [])
        if r.get("acronym") == acr
    )

# Reuse your earlier helpers if you have them:
def keys(d: dict) -> set[str]:
    return set(d["unique_acronyms"].keys())

def counts_by_acronym(d: dict) -> dict[str, int]:
    out = {}
    for o in d["occurrences"]:
        out[o["acronym"]] = out.get(o["acronym"], 0) + 1
    return out

# Minimal checker (reuse your existing `check` if defined)
PASSED = 0
FAILED = 0
def check(name: str, fn):
    global PASSED, FAILED
    try:
        fn()
        print(f"✅ {name}")
        PASSED += 1
    except AssertionError as e:
        print(f"❌ {name}: {e}")
        FAILED += 1
    except Exception as e:
        print(f"💥 {name}: {type(e).__name__}: {e}")
        FAILED += 1


# ================== NEW TESTS ==================

from pprint import pprint
def test_pto_three_senses_and_disambiguation():
    text = (
        "Please turn over (PTO) to continue.\n"
        "Our company offers generous Paid Time Off (PTO).\n"
        "The tractor’s power take-off (PTO) shaft needs grease."
    )
    d = detect_full(text)
    pprint(f"detected full{d}")

    # 1) detected once as a unique key, with three occurrences
    assert "PTO" in keys(d), f"unique keys: {keys(d)}"
    occ_counts = counts_by_acronym(d)
    assert occ_counts.get("PTO", 0) == 3, f"occurrence counts: {occ_counts}"

    # 2) three senses harvested
    sids = sense_ids(d["extraction"], "PTO")

    expected = {
        "pto|please_turn_over",
        "pto|paid_time_off",
        "pto|power_take_off",
    }
    assert expected.issubset(sids), f"senses seen: {sids}"

    # 3) each occurrence should resolve to one of the three
    chosen = chosen_ids_for_acr(d["extraction"], "PTO")
    # ensure all three show up among chosen (strong contexts near each PTO)
    assert expected.issubset(chosen), f"chosen set: {chosen}"

def test_ada_two_senses_and_one_selection_per_occurrence():
    text = (
        "The Americans with Disabilities Act (ADA) mandates accessibility. "
        "Meanwhile, the American Dental Association (ADA) publishes clinical guidance."
    )
    d = detect_full(text)
    assert "ADA" in keys(d)

    sids = sense_ids(d["extraction"], "ADA")
    want = {
        "ada|americans_with_disabilities_act",
        "ada|american_dental_association",
    }
    assert want.issubset(sids), f"senses: {sids}"

    chosen = [r for r in d["extraction"]["resolutions"] if r["acronym"] == "ADA"]
    assert len(chosen) == 2, f"resolutions: {chosen}"
    # both occurrences should be confidently assigned (contexts are clean)
    assert all(r["chosen_sense_id"] in want for r in chosen), f"chosen: {chosen}"
    # and they should not both pick the same sense
    assert len(set(r["chosen_sense_id"] for r in chosen)) == 2, f"chosen: {chosen}"

def test_am_defined_vs_time_of_day_undecided():
    text = (
        "AM stands for amplitude modulation.\n"
        "We meet at 10:30 AM tomorrow."
    )
    d = detect_full(text)
    assert "AM" in keys(d)
    occ_counts = counts_by_acronym(d)
    assert occ_counts.get("AM", 0) == 2

    # only one definitional sense exists
    sids = sense_ids(d["extraction"], "AM")
    assert "am|amplitude_modulation" in sids, f"senses: {sids}"

    # Expect one chosen = amplitude_modulation, and one undecided (None) for time-of-day
    chosen = [r for r in d["extraction"]["resolutions"] if r["acronym"] == "AM"]
    chosen_ids = [r["chosen_sense_id"] for r in chosen]
    assert "am|amplitude_modulation" in chosen_ids, f"chosen_ids: {chosen_ids}"
    assert any(cid is None for cid in chosen_ids), f"chosen_ids: {chosen_ids}"

# -------- run them --------
tests = [
    ("PTO: three senses + disambiguation", test_pto_three_senses_and_disambiguation),
    ("ADA: two senses, one per occurrence", test_ada_two_senses_and_one_selection_per_occurrence),
    ("AM: definition vs time-of-day undecided", test_am_defined_vs_time_of_day_undecided),
]
for name, fn in tests:
    check(name, fn)

print(f"\nSummary: {PASSED} passed, {FAILED} failed")


In [ ]:
from typing import Any
# --- longer notebook tests (no pytest needed) ---
import sys, pathlib, json, random, textwrap, itertools as it

# ensure package importable in notebook
try:
    import plainera_unacronym  # noqa
except ModuleNotFoundError:
    repo_root = pathlib.Path().resolve()
    sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))

def detect(text: str, **kwargs) -> Any:
    return detect_and_extract(text)

PASSED = 0
FAILED = 0

def check(name: str, fn):
    global PASSED, FAILED
    try:
        fn()
        print(f"✅ {name}")
        PASSED += 1
    except AssertionError as e:
        print(f"❌ {name}: {e}")
        FAILED += 1

def keys(d: dict) -> set[str]:
    return set(d["unique_acronyms"].keys())

def count_by_key(d: dict) -> dict[str, int]:
    out = {}
    for o in d["occurrences"]:
        out[o["acronym"]] = out.get(o["acronym"], 0) + 1
    return out


# 1) Long paragraph with multiple signals (defs, separators, mixed alnum, distractors)
def test_long_mixed_paragraph():
    text = textwrap.dedent("""
        At 10:30 AM we met the NHS analytics team. IT (Information Technology) owns LDAP and SSO.
        The R & D unit collaborates with the GPU cluster for H2O simulations and MP3 decoding benchmarks.
        OK, let's add O’RAN to the agenda alongside USB-C adapters. Later, AM stands for amplitude modulation in RF.
        We will avoid dotted initialisms like U.S. here and noisy tokens like C++ or A+B which are not acronyms.
    """).strip()
    d = detect(text)
    ks = keys(d)
    # keep
    assert "NHS" in ks
    assert "IT" in ks and d["unique_acronyms"]["IT"]["confidence"] >= 0.72
    assert "R&D" in ks  # normalized from "R & D"
    assert "GPU" in ks and "H2O" in ks and "MP3" in ks
    assert "O'RAN" in ks and "USB-C" in ks
    # drop
    assert "OK" not in ks           # interjection
    assert "AM" in ks               # kept because of definitional sentence later ("AM stands for ...")
    assert "US" not in ks           # dotted variant shouldn't be matched by your pattern
    # sanity: C++ and A+B not treated as acronyms
    assert all(acr not in {"C++", "A+B"} for acr in ks)

# 2) Very long distance between token and definition should NOT boost (directional + windowed)
def test_directional_window_limit():
    filler = " lorem ipsum dolor sit amet," * 20  # ~600+ chars
    text = f"IT {filler} stands for Information Technology."  # 'stands for' too far to the right
    d = detect(text)
    assert "IT" not in keys(d), "IT should not be boosted by far-away 'stands for'"

# 3) First-occurrence mapping remains stable with normalization
def test_first_occurrence_normalization_stable():
    text = "R & D met R&D after lunch. R & D then emailed."
    d = detect(text)
    assert "R&D" in keys(d)
    first = d["unique_acronyms"]["R&D"]
    # The first span should be the earliest occurrence in text
    assert first["start_offset"] == text.index("R & D")

# 4) Repetition & counts in a larger synthetic corpus
def test_large_repetition_counts():
    base = "NHS and R&D work with GPU and USB-C. "
    text = base * 250  # 250 repetitions
    d = detect(text)
    counts = count_by_key(d)
    # Expect ~250 occurrences each (some tokens might appear twice per base, adjust as needed)
    for acr in ["NHS", "R&D", "GPU", "USB-C"]:
        assert acr in counts and counts[acr] >= 230, f"{acr} count too low: {counts.get(acr)}"

# 5) Hyphen/en-dash noise: GPU should still be detected; right-hand lower token shouldn't be needed
def test_gpu_with_following_dash_word():
    text = "We tested GPU–accelerated pipelines and GPU-accelerated kernels yesterday."
    d = detect(text)
    ks = keys(d)
    assert "GPU" in ks, "GPU should be detected even when followed by dash-word"
    # Ensure we didn't create a weird token spanning the dash
    assert all(o["acronym"] != "GPU–accelerated" for o in d["occurrences"])

# 6) Company suffixes and short common uppers drop (unless defined)
def test_company_suffixes_drop_and_ok_drops():
    text = "Acme LTD acquired Example PLC. OK, moving on. DR Smith arrived at 7 AM."
    d = detect(text)
    ks = keys(d)
    assert "LTD" not in ks and "PLC" not in ks
    assert "OK" not in ks
    assert "AM" not in ks  # time-of-day
    # DR is in non_acronym_upper → drop unless explicit definition
    assert "DR" not in ks

# 7) Parenthetical definition rescues otherwise noisy tokens
def test_parenthetical_rescue_for_ok_and_short_tokens():
    text = "OK (Object Kernel) appeared in legacy docs; AI (Artificial Intelligence) and IT (Information Technology) led."
    d = detect(text)
    ks = keys(d)
    assert "OK" in ks and d["unique_acronyms"]["OK"]["confidence"] >= 0.72
    assert "AI" in ks and d["unique_acronyms"]["AI"]["confidence"] >= 0.72
    assert "IT" in ks

# 8) Parallel == serial parity on a long doc
def test_parallel_parity_long_doc():
    para = (
        "We’ll loop in R&D after the NHS workshop. IT (Information Technology) leads. "
        "O’RAN and USB-C were discussed. GPU outperformed CPU. "
    )
    big = para * 300
    serial = detect(big, parallel=False)
    parallel = detect(big, parallel=True)
    assert keys(serial) == keys(parallel)
    assert count_by_key(serial) == count_by_key(parallel)

# 9) Context window snaps to sentence boundaries (lenient check)
def test_context_window_sentence_bounds():
    text = "Alpha. We met the NHS team today and they agreed. Beta."
    d = detect(text)
    nhs_occ = next(o for o in d["occurrences"] if o["acronym"] == "NHS")
    left, right = nhs_occ["context_window"]
    segment = text[left:right]
    assert segment.strip().endswith("agreed."), f"window right seems off: {segment!r}"
    assert segment.strip().startswith("We met the"), f"window left seems off: {segment!r}"

# 10) Long definitional form lines (parentheses) still boost within limit
def test_long_parenthetical_still_boosts_within_limit():
    long_def = "(Information Technology and related shared infrastructure services)"
    text = f"IT {long_def} owns the platform."
    d = detect(text)
    assert "IT" in keys(d) and d["unique_acronyms"]["IT"]["confidence"] >= 0.72


# --- run all ---
tests = [
    ("Long mixed paragraph", test_long_mixed_paragraph),
    ("Directional window limit", test_directional_window_limit),
    ("First-occurrence normalization", test_first_occurrence_normalization_stable),
    ("Large repetition counts", test_large_repetition_counts),
    ("GPU with following dash word", test_gpu_with_following_dash_word),
    ("Company suffixes & OK drop", test_company_suffixes_drop_and_ok_drops),
    ("Parenthetical rescue for short tokens", test_parenthetical_rescue_for_ok_and_short_tokens),
    ("Parallel parity on long doc", test_parallel_parity_long_doc),
    ("Context window sentence bounds", test_context_window_sentence_bounds),
    ("Long parenthetical boost", test_long_parenthetical_still_boosts_within_limit),
]

PASSED = FAILED = 0
for name, fn in tests:
    check(name, fn)

print(f"\nSummary: {PASSED} passed, {FAILED} failed")


In [1]:
# ---- big paragraphs: stress test in a single notebook cell ----
import sys, pathlib, json, textwrap
from plainera_unacronym.nlp.execute import detect_and_extract
from pprint import pprint as pp
# Ensure package importable in the notebook
try:
    import plainera_unacronym  # noqa
except ModuleNotFoundError:
    repo_root = pathlib.Path().resolve()
    sys.path.insert(0, str(repo_root / "packages" / "plainera_unacronym" / "src"))




def keys(d: dict) -> set[str]:
    return set(d["unique_acronyms"].keys())


def counts_by_key(d: dict) -> dict[str, int]:
    c = {}
    for o in d["occurrences"]:
        c[o["acronym"]] = c.get(o["acronym"], 0) + 1
    return c


# ---- long, mixed-content paragraphs ----
LONG_TEXT = textwrap.dedent("""
    The American Psychological Association (APA) publishes influential journals, while the American Planning Association (APA) guides urban development.
    In finance, the Consumer Price Index (CPI) tracks inflation, whereas in computing CPI often refers to cycles per instruction.
    Single Sign-On (SSO) simplifies access, and SSO stands for Single Sign-On across most IT platforms. Lightweight Directory Access Protocol (LDAP) integrates
    with Transport Layer Security (TLS) for secure binds in many NHS trusts. Jacob says, ALRIGHTY THEN!

    The Americans with Disabilities Act (ADA) ensures accessibility, but the American Dental Association (ADA) sets clinical guidelines.
    Corporate Social Responsibility (CSR) shapes strategy, whereas a Certificate Signing Request (CSR) kicks off PKI workflows.
    We ran GPU–accelerated jobs and compared GPU results to CPU baselines; R&D will review them alongside I/O traces and S&P 500 sector notes.

    The European Medicines Agency (EMA) approves drugs in the EU, while an exponential moving average (EMA) is a trading indicator.
    Centers for Disease Control and Prevention (CDC) issue guidance; in data engineering, Change Data Capture (CDC) drives downstream updates.
    Return on Investment (ROI) guides budgets, but Region of Interest (ROI) guides image processing.

    The Department of Energy (DOE) funds basic research, while Design of Experiments (DOE) structures trials.
    The Securities and Exchange Commission (SEC) oversees markets; the Southeastern Conference (SEC) organizes college sports.
    The Central Processing Unit (CPU) remains a baseline as Graphics Processing Units (GPU) scale out; Random Access Memory (RAM) capacity still gates workloads.
    “IT was tricky to reproduce” is just a sentence start, but IT (Information Technology) owns the SSO/LDAP stack.

    Digital Subscriber Line (DSL) brought early broadband; a Domain-Specific Language (DSL) made our pipeline concise.
    A Peripheral Component Interconnect (PCI) slot differs from Payment Card Industry (PCI) compliance.
    Earnings Per Share (EPS) moved after ISO-certified audits; Encapsulated PostScript (EPS) assets rendered crisply.
    We exported JSON and CSV snapshots; H2O chemistry demos stayed separate from MP3 decoding tests.

    The International Telecommunication Union (ITU) sets standards; the International Triathlon Union (ITU) runs competitions.
    The National Archives and Records Administration (NARA) preserves documents; the North American Retail Association (NARA) advocates for merchants.
    O’RAN specs advanced; USB-C hubs shipped. OK, we’ll regroup at 10:45 AM, and PM (Project Manager) will chair; later, PM stands for particulate matter.

    The British Standards Institution (BSI) audits suppliers; Business Systems Integration (BSI) teams coordinate ERP rollouts.
    Quality Assurance (QA) wrote test plans; Quality Control (QC) validated outputs.
    User Experience (UX) and User Interface (UI) workshops ran back-to-back; Estimated Time of Arrival (ETA) for the next build is 18:30.
    Note that U.S. and U.K. dotted forms appear here but aren’t acronyms under our pattern.

    The World Wide Web Consortium (W3C) advanced specs; HyperText Markup Language (HTML) docs and Cascading Style Sheets (CSS) were version-locked.
    The International Organization for Standardization (ISO) reviewed findings; the Insurance Services Office (ISO) published actuarial updates.
    The Federal Communications Commission (FCC) ruled on spectrum; Field-Programmable Gate Arrays (FPGA) sped up TLS offload.

    The International Criminal Court (ICC) issued guidance; in sports, the International Cricket Council (ICC) scheduled fixtures.
    The European Central Bank (ECB) raised rates; the Electronic Code Book (ECB) mode remained deprecated in crypto courses.
    Meanwhile, the North Atlantic Treaty Organization (NATO) met with the National Oceanic and Atmospheric Administration (NOAA) about satellite data.
    NASA’s outreach continues, while NASA events celebrate saxophone music in the North American Saxophone Alliance (NASA).
""").strip()

tester = """The International Criminal Court (ICC) issued guidance; in sports, the International Cricket Council (ICC) scheduled fixtures.
    The European Central Bank (ECB) raised rates; the Electronic Code Book (ECB) mode remained deprecated in crypto courses.""".strip()

tester_2 = (
    "The European Central Bank (ECB) raised rates; "
    "In sports, the International Cricket Council (ICC) scheduled fixtures. "
    "The World Wide Web Consortium (W3C) advanced specs; "
    "HyperText Markup Language (HTML) docs and Cascading Style Sheets (CSS) were version-locked. "
    "The Federal Communications Commission (FCC) ruled on spectrum; "
    "Field-Programmable Gate Arrays (FPGA) sped up TLS offload. "
    "Later, the International Criminal Court (ICC) issued further guidance. "
    "the Electronic Code Book (ECB) mode remained deprecated in crypto courses."
)

def run_long_text_test():
    det, extr, report = detect_and_extract(tester_2, return_reports=True)
    pp(det)
    pp(extr)
    pp(report)

run_long_text_test()


DetectorResult(unique_acronyms={'CSS': FirstOccurrence(acronym='CSS',
                                                       start_offset=235,
                                                       end_offset=238,
                                                       occurrence_confidence=0.85,
                                                       normalized_key='CSS'),
                                'ECB': FirstOccurrence(acronym='ECB',
                                                       start_offset=27,
                                                       end_offset=30,
                                                       occurrence_confidence=0.85,
                                                       normalized_key='ECB'),
                                'FCC': FirstOccurrence(acronym='FCC',
                                                       start_offset=300,
                                                       end_offset=303,
                                      

In [ ]:
def test_shouty_phrase_drops():
    text = "Jacob says, ALRIGHTY THEN! We’ll reconvene."
    d = detect(text)
    ks = keys(d)
    assert "ALRIGHTY" not in ks, f"shout interjection leaked: {ks}"
    assert "THEN" not in ks, f"shout tail leaked: {ks}"

check("Shouty ALL-CAPS phrase drops", test_shouty_phrase_drops)


In [ ]:
def test_shout_rule_does_not_kill_real_acronym():
    d = detect("Big news: NASA! launches today.")
    assert "NASA" in keys(d), "NASA wrongly dropped by shout rule"

check("Shout rule doesn’t kill real acronyms", test_shout_rule_does_not_kill_real_acronym)


In [ ]:
import importlib

from plainera_unacronym.nlp.common.types import DetectorConfig, pattern_cache
from plainera_unacronym.nlp.detection.detector import Detector
import plainera_unacronym.nlp.detection.heuristics.core as core
importlib.reload(core)         # ensure we have the latest compile_pattern
pattern_cache.clear()          # avoid sta
# le compiled patterns

In [ ]:
cfg = DetectorConfig(enable_dotted=True)
pat = core.compile_acronym_pattern(cfg)
print("PATTERN:", pat.pattern)

assert "(?:[A-Z]\\.){2,}" in pat.pattern, "Dotted branch missing in pattern"
assert "(?:[A-Z][a-z]?){2,5}" in pat.pattern, "CamelCaps branch missing in pattern"

In [ ]:

txt = "The U.S. economy and U.K. policy differ. NASA leads."
cfg = DetectorConfig(enable_dotted=True, dotted_display='preserve')  # case-preserving for mixed-case is fine
d = Detector(cfg).detect(txt)
print(list(d.unique_acronyms.keys()))

In [ ]:
txt_dotted = "The U.S. economy and U.K policy differ. NASA leads."
dotted = Detector(DetectorConfig(enable_dotted=True)).detect(txt_dotted)
print("DOTTED keys:", list(dotted.unique_acronyms.keys()))
# If this fails, your normalize_key call isn't stripping dots.
assert "US" in dotted.unique_acronyms, dotted.unique_acronyms
assert "UK" in dotted.unique_acronyms, dotted.unique_acronyms

In [ ]:
txt_mixed = "Transport for London (TfL) runs the Tube. TfL operates buses."
mixed_on  = Detector(DetectorConfig(enable_mixed_case=True)).detect(txt_mixed)
mixed_off = Detector(DetectorConfig(enable_mixed_case=False)).detect(txt_mixed)
print("MIXED ON keys:", list(mixed_on.unique_acronyms.keys()))
print("MIXED OFF keys:", list(mixed_off.unique_acronyms.keys()))

# If this fails, either the camel branch isn't active or keys aren't uppercased.
assert "TfL" in mixed_on.unique_acronyms, mixed_on.unique_acronyms
assert "TFL" not in mixed_off.unique_acronyms, mixed_off.unique_acronyms

In [ ]:
from plainera_unacronym.nlp.detection.detector import Detector
from plainera_unacronym.nlp.common.types import DetectorConfig

def keys(d): return set(d.unique_acronyms.keys())

# 1) Separators + dotted
txt = 'R & D met R&D. USB-C and O’RAN followed. U.S. policy differs. NHS) ok.'
on  = Detector(DetectorConfig(enable_dotted=True)).detect(txt)
off = Detector(DetectorConfig(enable_dotted=False)).detect(txt)
assert "R&D" in keys(on) and "USB-C" in keys(on) and "O'RAN" in keys(on)
assert "US" in keys(on) and "US" not in keys(off)




In [ ]:
# 2) Mixed-case
txt2 = "Transport for London (TfL) runs the Tube. TfL operates buses."
mc_on  = Detector(DetectorConfig(enable_mixed_case=True)).detect(txt2)
mc_off = Detector(DetectorConfig(enable_mixed_case=False)).detect(txt2)
assert "TfL" in keys(mc_on)
assert "TFL" not in keys(mc_off)

In [ ]:
from plainera_unacronym.wiring.composition import sink
# 1) Ensure bio plugin registers with the global registry
import plainera_unacronym.core.domains.bio.plugin  # noqa: F401  <-- adjust path if yours differs

from dataclasses import replace
from plainera_unacronym.nlp.detection.domains.bio.plugin import BioPlugin
from plainera_unacronym.nlp.detection.detector import Detector
from plainera_unacronym.nlp.common.types import DetectorConfig

def keys(d):
    return set(d.unique_acronyms.keys())

def test_bio_plugin_detects_greek_utr_and_virus():
    txt = "Measured IFN-γ and mRNA in SARS-CoV-2 5′-UTR; IL-6 (95% CI 1.2–2.3). USA U.K."

    # Enable the bio domain (merge, don't replace existing domains)
    # cfg = replace(
    #     DetectorConfig(enable_dotted=True),
    #     enabled_domains=frozenset({"bio"}),
    #     domain_cfg={"bio": BioConfig()},
    # )

    det = Detector(cfg)
    res = det.detect(txt)
    ks = keys(res)
    print(ks)

    # Bio-y hits
    assert {"IFN-γ", "SARS-CoV-2", "IL-6"}.issubset(ks)
    # UTR may normalize primes; be flexible
    assert any("UTR" in k for k in ks)

# 2) Run it
test_bio_plugin_detects_greek_utr_and_virus()

In [ ]:
# Notebook cell 2
async def _ticker(duration=0.25, period=0.02):
    """Ticks while detection runs; if ticks>0 the loop wasn't blocked."""
    ticks = 0
    start = time.perf_counter()
    while time.perf_counter() - start < duration:
        await asyncio.sleep(period)
        ticks += 1
    return ticks


In [ ]:
from plainera_unacronym.nlp.detection.domains.bio.config import BioConfig
import asyncio, time, threading
from dataclasses import dataclass
from plainera_unacronym.nlp.common.types import DetectorConfig
from plainera_unacronym.nlp.detection.detector import Detector
@dataclass
class DetectorResult:
    status: str
    worker_thread: str

cfg = replace(
    DetectorConfig(enable_dotted=True),
    enabled_domains=frozenset({"bio"}),
    domain_cfg={"bio": BioConfig()},
)

detector = Detector(cfg)

async def _ticker(duration=0.35, period=0.05):
    """Runs while detect_async is working to prove the loop isn't blocked."""
    ticks = 0
    start = time.perf_counter()
    while time.perf_counter() - start < duration:
        await asyncio.sleep(period)
        ticks += 1
    return ticks


In [ ]:
# Notebook cell 3
async def run_test():
    # Give it some work so timing/ticker are meaningful.
    base = "NASA will launch SLS with the ESA service module."
    text = " ".join([LONG_TEXT] * 30)  # enlarge input to simulate heavier work

    # --- Monkey-patch detect_parallel to record the worker thread name ---
    orig = detector.detect_parallel
    detector._last_worker_thread = None  # probe field

    def spy(text_arg):
        detector._last_worker_thread = threading.current_thread().name
        return orig(text_arg)

    detector.detect_parallel = spy  # patch on the instance

    try:
        # Run async wrapper + a ticker concurrently
        t0 = time.perf_counter()
        result_task = asyncio.create_task(detector.detect_async(text))
        ticks_task  = asyncio.create_task(_ticker())
        result, ticks = await asyncio.gather(result_task, ticks_task)
        elapsed = time.perf_counter() - t0

        # Also compute expected via the original sync function
        expected = orig(text)

        # ---- Assertions ----
        # 1) loop not blocked
        assert ticks > 0, f"event loop looked blocked (ticks={ticks})"

        # 2) results match (compare cheap invariants to avoid huge prints)
        assert len(result.occurrences) == len(expected.occurrences)
        assert len(result.unique_acronyms) == len(expected.unique_acronyms)

        # 3) ran off the loop thread (spy captured the worker thread)
        assert detector._last_worker_thread is not None
        assert detector._last_worker_thread != threading.current_thread().name, (
            f"detect_parallel ran on the loop thread: {detector._last_worker_thread}"
        )

        print(f"✅ non-blocking: {ticks} ticks while waiting (~{elapsed:.3f}s)")
        print(f"✅ results match: {len(result.occurrences)} occs / {len(result.unique_acronyms)} uniques")
        print(f"✅ ran in background thread: {detector._last_worker_thread}")

    finally:
        # restore the original method
        detector.detect_parallel = orig
        if hasattr(detector, "_last_worker_thread"):
            delattr(detector, "_last_worker_thread")

# In Jupyter you can await at top level:
await run_test()


In [ ]:
from pprint import pprint as pp
from plainera_unacronym.nlp.execute import detect_and_extract

text2 = (
    "Natural language processing (NLP) is used for analysing text and language models. "
    "Nice Lovely Plants (NLP) are sold in the garden centre. "
    "Later, NLP was watered daily and grew quickly in the greenhouse."
)


# Disabled

_det0, extr0, r0 = detect_and_extract(text2, return_reports=True)
for res in extr0.resolutions:
    if res.acronym.upper() == "NLP":
        print(res.start, res.chosen_sense_id, res.margin, res.gap)

# pp(_det0)
# pp(extr0)
pp(r0)

In [ ]:
text1 = (
    "Graphics Processing Unit (GPU) accelerates kernel execution on the device. "
    "General Purpose Unit (GPU) is a generic term in another department. "
    "Later, the GPU was saturated due to kernel launch overhead on the device."
)
text2 = (
    "Natural language processing (NLP) is used for analysing text and language models. "
    "Nice Lovely Plants (NLP) are sold in the garden centre. "
    "Later, NLP was watered daily and grew quickly in the greenhouse."
)
text3 = (
    "European Medicines Agency (EMA) issued guidance on medicines and clinical trials. "
    "Email Marketing Automation (EMA) increased click-through rates for campaigns. "
    "Later, EMA published new guidance about clinical trial reporting."
)
text4 = (
    "Application Programming Interface (API) allows clients to integrate with services. "
    "Active Pharmaceutical Ingredient (API) must meet purity standards in manufacturing. "
    "Later, the API returned a JSON response after the client retried the request."
)
text5 = (
    "Polymerase chain reaction (PCR) was used to amplify DNA in the lab. "
    "Project Change Request (PCR) was raised to adjust the delivery timeline. "
    "Later, PCR results were validated and the DNA bands were visible."
)
text_pharma_app = (
    """
 On Monday the delivery team received a short PDF (Portable Document Format) briefing titled “Telemetry Uplift”. It described a new API (Application Programming Interface) for uploading device metrics, plus a note that the GPU (Graphics Processing Unit) on the inference box should be enabled for faster embeddings. The PMO (Project Management Office) forwarded the PDF again two hours later, asking whether the API could be “production-ready by Friday” and whether the GPU change would affect cost. We replied that the API contract looked stable, the PDF was clear enough, and the PMO could expect an updated timeline after lunch.

After lunch, the data scientist skimmed the same PDF but got stuck on a sentence about “the PDF of latency errors”. In that section the author clearly meant PDF (Probability Density Function), because they immediately discussed a skewed distribution and suggested fitting a mixture model. The scientist annotated the chart and said the PDF should be recalculated once the API starts emitting more samples; otherwise the tail behaviour would be misleading. Someone then asked if the GPU was “needed for the PDF work”, which made sense in the statistics context (accelerating vector operations), even though the earlier GPU mention in the infrastructure section was about inference throughput.

On Tuesday, a compliance reviewer raised a different issue: the API was also mentioned in an attached manufacturing appendix as API (Active Pharmaceutical Ingredient). The appendix wasn’t software at all—it referenced batch records, assay limits, and a supplier certificate for the API. The reviewer wrote, “If the API provenance is incomplete, we can’t sign off,” and the developer almost replied with a link to the API docs before realising the context was medicinal chemistry, not HTTP endpoints. To reduce confusion, the PMO asked that “the API” be referenced with clearer wording in future minutes, but the note still came through as: “Confirm API status; update PDF.”

On Wednesday, the team travelled to an airport lab to validate field connectivity. The site engineer pointed at a trolley and said the GPU (Ground Power Unit) must be connected before the avionics rack would boot. A visiting developer, still thinking about GPUs and embeddings, asked whether the GPU supported CUDA. The engineer stared for a beat, then explained the GPU provides external electrical power to the aircraft systems on the stand, and suggested we “not plug laptops into it unless you enjoy paperwork”. Later, the same developer wrote in the report that the GPU should be available during testing; the PMO read that line and assumed it was a compute request, not ground equipment.

By Thursday, a news alert came in about the PMO (Prime Minister’s Office) issuing a statement on data sharing. Our PMO (Project Management Office) promptly asked whether “the PMO guidance” changed our API design, and whether the PDF needed updating. The lawyer replied that the PMO statement was political context, not project governance, but it still touched on privacy expectations that might affect what the API emits. The analyst then summarised: update the PDF (Portable Document Format) spec, regenerate the PDF (Probability Density Function) plots with new samples, confirm the API (Application Programming Interface) schema, and keep the API (Active Pharmaceutical Ingredient) appendix separate. Everyone agreed—mostly because nobody wanted a fifth meaning of GPU to appear in the next meeting.

    """
)

text_ema = (
    """
    European Medicines Agency (EMA) published updated safety guidance today.
    Our Email Marketing Automation (EMA) system sends the weekly newsletter to subscribers.
    Later, the EMA issued a warning related to pharmacovigilance reporting.
    """
)



In [ ]:
from plainera_unacronym.nlp.execute import detect_and_extract

def run_case(text):
    det, extr, reports, state = detect_and_extract(text, return_reports=True, return_state=True)
    print(state.disambig.tier2.report)
    print(state.disambig.tier2.ranked)

    for r2 in state.disambig.tier2.ranked:
        if r2.applied:
            print(r2.occ, r2.tier2_sims, r2.blended_scores)

    print("tier2 report:", getattr(getattr(extr, "tier2_report", None), "reasons", None))
    print("---- stage reports ----")
    for r in reports:
        if r.name in ("tier1_score_occurrences", "tier2_semantic_rerank", "tier1_select_and_assemble"):
            print(r.name, "->", r.info)

    print("---- ambiguous keys ----", extr.ambiguous_keys)

    print("---- resolutions ----")
    for res in extr.resolutions:
        if res.acronym.upper() in {k.upper() for k in extr.ambiguous_keys}:
            print(res.acronym, res.start, res.chosen_sense_id, "margin=", res.margin, "gap=", res.gap)

    print("\n\n\n***************CEILING************************\n\n\n")
    ceiling = 0.25
    for r in extr.resolutions:
        if r.acronym.upper() in extr.ambiguous_keys and r.margin < ceiling:
            print(r.acronym, r.start, "margin=", r.margin, "chosen=", r.chosen_sense_id)


texts = [text1, text2, text3, text4, text5, text_pharma_app, text_ema]
texts_2 = [text_pharma_app]
for  text in texts_2:
    print("***************************")
    run_case(text)
    print("***************************")


